<a href="https://colab.research.google.com/github/anangshachatterjee/anangsha-chatterjee-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one report-date observation for one pseudonymized client and one pseudonymized content item.

**Primary table:** `fact_content_daily_performance`.

**Development time window:** March 2026 (`month=2026-03`).

I will use March 2026 as the mid-panel development month. I will not use the final-month `_sample` for developing label logic because it represents the latest full month and will be treated as a sealed test period.

**Prediction/ranking goal:** I want to rank content pages by their likelihood of showing declining search performance, using only information available before the decision moment.

**Deliberate exclusion:** I will exclude fields derived from the future outcome or fields that would only be available after the decision, because they could introduce target leakage.

In [2]:
from google.colab import userdata
from huggingface_hub import HfApi, HfFileSystem

HF_TOKEN = userdata.get("HF_TOKEN")

# Test that the token can access the dataset
api = HfApi(token=HF_TOKEN)
info = api.dataset_info("FlyRank/internship-warehouse")

print("Dataset access confirmed:", info.id)

Dataset access confirmed: FlyRank/internship-warehouse


In [3]:
!pip -q install -U huggingface_hub duckdb
from huggingface_hub import HfFileSystem
import duckdb

# Create filesystem using your authenticated token
fs = HfFileSystem(token=HF_TOKEN)

# Register the authenticated filesystem with DuckDB
duckdb.register_filesystem(fs)

print("Authenticated Hugging Face filesystem registered.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 92.2 MB/s eta 0:00:00
Authenticated Hugging Face filesystem registered.


In [4]:
files = fs.glob(
    "datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

print("Number of March 2026 files:", len(files))
print(files[:5])

Number of March 2026 files: 1
['datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet']


In [5]:
con = duckdb.connect()

print("New DuckDB connection created.")

New DuckDB connection created.


In [6]:
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

files = list(
    api.list_repo_tree(
        repo_id="FlyRank/internship-warehouse",
        path_in_repo="fact_content_daily_performance/month=2026-03",
        repo_type="dataset",
        recursive=False
    )
)

for f in files:
    print(f.path)

fact_content_daily_performance/month=2026-03/data_0.parquet


In [8]:
for f in files:
    print(f.path, "→", f.size / (1024**3), "GB")

fact_content_daily_performance/month=2026-03/data_0.parquet → 0.11570165865123272 GB


In [12]:
try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("✅ Dataset access confirmed")
    print(info.id)
except Exception as e:
    print("❌ Dataset access problem")
    print(type(e).__name__, str(e))

✅ Dataset access confirmed
FlyRank/internship-warehouse


In [15]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Secret loaded:", bool(HF_TOKEN))
print("Starts with hf_:", HF_TOKEN.startswith("hf_") if HF_TOKEN else False)
print("Token length:", len(HF_TOKEN) if HF_TOKEN else 0)

Secret loaded: True
Starts with hf_: True
Token length: 37


In [16]:
from huggingface_hub import hf_hub_download

local_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Downloaded successfully:")
print(local_file)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Downloaded successfully:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [17]:
import duckdb

con = duckdb.connect()

test = con.sql(
    f"""
    SELECT *
    FROM read_parquet('{local_file}')
    LIMIT 5
    """
).df()

test

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [18]:
schema = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{local_file}')
    """
).df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



### Features

I will use historical search-performance variables that were available before the decision moment. I will keep the feature set to a maximum of five variables.

### Label

The label/proxy will represent a future decline in search performance. It will be defined from an outcome window after the feature/decision window.

### Context

Pseudonymized client and content identifiers will be used for grouping, joining, and verification rather than treated as meaningful predictive signals.

### Excluded

I will exclude fields that directly encode the outcome, are calculated from the future window, or would only be known after the decision. These fields could cause target leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [26]:
grain_check = con.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(
            DISTINCT (
                report_date,
                client_hash_id,
                content_hash_id
            )
        ) AS distinct_grain
    FROM read_parquet('{local_file}')
    """
).df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,distinct_grain
0,9841378,9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



### Limitation: Unequal historical coverage

The warehouse is an unbalanced panel, so different clients have different depths of historical data. This means that observations from different clients may not represent equally complete histories.

The data is observational rather than experimental. It can show measured patterns and support ranking decisions, but it cannot by itself prove that a particular content change caused a future improvement in search performance.

The fixed 90-day query windows can also overlap reporting periods, so feature and outcome windows must be defined carefully to avoid using future information.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.